# 🎵 MusicGen Fine-Tuning Journey
## Notebook 3: Fine-Tuning

---

### What we're doing

This is the core of the project. We're going to train MusicGen's transformer to produce better throat singing by:
1. **Loading MusicGen-small** (our starting point)
2. **Freezing EnCodec** — we do NOT want to change how audio is tokenized, just how tokens are predicted
3. **Building a training loop** — feed (text, audio_tokens) pairs through the transformer, compute prediction error, backpropagate
4. **Saving checkpoints** along the way so you can recover from any point

### What fine-tuning actually does

The transformer LM has learned, from its original training, patterns like:
- "jazz" → high probability of jazz chord token sequences
- "drum loop" → high probability of percussion-like token patterns

It does NOT know:
- "khoomei" → the specific token sequences that EnCodec produces from throat singing audio

Fine-tuning shows it many examples of: *(text description of throat singing) → (encoded throat singing tokens)*  
...and adjusts the LM's weights slightly so that, when it sees these text prompts, it generates more throat-singing-like token sequences.

### The risk: Catastrophic Forgetting

If we train too aggressively (high LR, many epochs), the model will start to "forget" everything it learned about other music styles. We use:
- **Very low learning rate** (1e-5) — small steps, don't overwrite old knowledge
- **Few epochs** — just enough to shift the distribution, not overwrite it
- **Validation loss monitoring** — if val loss rises while train loss falls, we're overfitting


## Step 1: Load Config and Verify Dataset

In [ ]:
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm

# Load dataset config written by notebook 2
CONFIG_PATH = Path('dataset/dataset_config.json')

if not CONFIG_PATH.exists():
    print("❌ dataset/dataset_config.json not found!")
    print("   Make sure you ran Notebook 2 first.")
else:
    with open(CONFIG_PATH) as f:
        cfg = json.load(f)

    DATA_DIR       = Path(cfg['data_dir'])
    TRAIN_MANIFEST = Path(cfg['train_manifest'])
    VAL_MANIFEST   = Path(cfg['val_manifest'])
    SAMPLE_RATE    = cfg['sample_rate']
    CLIP_DURATION  = cfg['clip_duration']

    print("Dataset config loaded:")
    print(f"  Data dir:   {DATA_DIR}")
    print(f"  Train:      {cfg['n_train']} clips")
    print(f"  Val:        {cfg['n_val']} clips")
    print(f"  Sample rate:{SAMPLE_RATE} Hz")
    print(f"  Clip dur:   {CLIP_DURATION}s")

    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"\nDevice: {DEVICE}")

    if DEVICE == 'cuda':
        vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"VRAM:   {vram:.1f} GB")

## Step 2: Dataset Class

PyTorch's `Dataset` and `DataLoader` handle batching and shuffling for us. Our dataset:
- Reads the manifest JSONL files
- Loads WAV files on demand (not all at once — saves RAM)
- Returns `(audio_tensor, description_string)` pairs

The audio tensor shape will be `[1, T]` (1 channel, T samples), which is what EnCodec expects.

In [ ]:
import torch
import soundfile as sf
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


class ThroatSingingDataset(Dataset):
    """
    Simple dataset: reads (wav_path, description) pairs from a JSONL manifest.
    Returns fixed-length audio tensors + description strings.
    """

    def __init__(self, manifest_path: Path, data_dir: Path,
                 sample_rate: int = 32000, clip_duration: float = 10.0):
        self.data_dir = data_dir
        self.sample_rate = sample_rate
        self.max_samples = int(sample_rate * clip_duration)

        # Load all records from the JSONL file
        self.records = []
        with open(manifest_path) as f:
            for line in f:
                line = line.strip()
                if line:
                    self.records.append(json.loads(line))

        print(f"Dataset loaded: {len(self.records)} records from {manifest_path.name}")

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        record = self.records[idx]
        audio_path = self.data_dir / record['path']
        description = record['description']

        # Load audio using soundfile (avoids torchcodec issues)
        audio_np, sr = sf.read(str(audio_path))
        
        # Convert to tensor and ensure correct shape [C, T]
        audio = torch.from_numpy(audio_np).float()
        if audio.dim() == 1:
            audio = audio.unsqueeze(0)  # [T] -> [1, T]
        else:
            audio = audio.T  # [T, C] -> [C, T] if stereo

        # Resample if needed (shouldn't be necessary if notebook 2 worked correctly)
        if sr != self.sample_rate:
            import torchaudio
            audio = torchaudio.functional.resample(audio, sr, self.sample_rate)

        # Ensure mono [1, T]
        if audio.shape[0] > 1:
            audio = audio.mean(dim=0, keepdim=True)

        # Pad or truncate to exact length
        if audio.shape[-1] >= self.max_samples:
            audio = audio[:, :self.max_samples]
        else:
            pad_amount = self.max_samples - audio.shape[-1]
            audio = F.pad(audio, (0, pad_amount))

        return audio, description  # ([1, T], str)


# Instantiate both datasets
train_dataset = ThroatSingingDataset(TRAIN_MANIFEST, DATA_DIR, SAMPLE_RATE, CLIP_DURATION)
val_dataset   = ThroatSingingDataset(VAL_MANIFEST,   DATA_DIR, SAMPLE_RATE, CLIP_DURATION)

# Test one sample
audio_sample, desc_sample = train_dataset[0]
print(f"\nSample check:")
print(f"  Audio shape: {list(audio_sample.shape)}  (channels, samples)")
print(f"  Duration:    {audio_sample.shape[-1] / SAMPLE_RATE:.1f}s")
print(f"  Description: {desc_sample}")

In [ ]:
# ─── Training Hyperparameters ───────────────────────────────────────────
# These are sensible defaults for a small dataset fine-tuning run on a 5090.
# Explanations below each one.

BATCH_SIZE   = 4        # How many clips per gradient step.
                        # Larger = faster training, but more VRAM.
                        # On 5090 (24GB), 4-8 should be stable.

LEARNING_RATE = 1e-5   # How large each weight update is.
                        # 1e-5 is conservative — we want gentle nudges,
                        # not aggressive overwriting of existing knowledge.

NUM_EPOCHS   = 30       # How many times we see the full dataset.
                        # 30 epochs on ~100 clips is a mild fine-tune.
                        # Watch val loss — stop early if it starts rising.

GRAD_CLIP    = 1.0      # Clips gradient norm to prevent exploding gradients.
                        # Standard practice for transformer training.

SAVE_EVERY   = 5        # Save a checkpoint every N epochs.

CHECKPOINT_DIR = Path('checkpoints')
CHECKPOINT_DIR.mkdir(exist_ok=True)

# DataLoaders — handles batching + shuffling automatically
# num_workers=0 is safer in WSL2 (avoids multiprocessing issues)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, drop_last=False)

print(f"Train batches per epoch: {len(train_loader)}")
print(f"Val   batches per epoch: {len(val_loader)}")
print(f"\nWith {NUM_EPOCHS} epochs: {NUM_EPOCHS * len(train_loader)} total gradient steps")

## Step 3: Load MusicGen and Configure for Training

The key preparation steps:
1. Load the model as normal
2. **Freeze EnCodec** — `requires_grad = False` means no gradients flow through it, no updates happen
3. **Keep LM trainable** — `requires_grad = True` (default, just being explicit)
4. Only pass **LM parameters** to the optimizer — this is crucial

If you accidentally pass all parameters to the optimizer and EnCodec gradients somehow enable, you'll corrupt the codec and audio generation quality will tank.

In [ ]:
from audiocraft.models import MusicGen
from audiocraft.modules.conditioners import ConditioningAttributes
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

print("Loading MusicGen-small...")
model = MusicGen.get_pretrained('facebook/musicgen-small', device=DEVICE)

# ─── CRITICAL: Fix dtype mismatch ──────────────────────────────────────
# audiocraft 1.3.0 loads some layers as float16 and others as float32.
# This causes NaN during training. Convert entire LM to float32.
print("Converting model to float32 for training stability...")
model.lm = model.lm.float()  # Convert all LM parameters to float32

# Verify dtypes are consistent
sample_emb = next(model.lm.emb.parameters())
sample_transformer = next(model.lm.transformer.parameters())
print(f"  LM embedding dtype: {sample_emb.dtype}")
print(f"  LM transformer dtype: {sample_transformer.dtype}")

# ─── Freeze EnCodec ────────────────────────────────────────────────────
# EnCodec = the audio tokenizer. We do NOT want to change this.
# If we changed it, the tokens would shift meaning, breaking everything.
for param in model.compression_model.parameters():
    param.requires_grad = False
model.compression_model.eval()  # Also set to eval mode — no dropout etc.

# ─── Train only the Language Model ─────────────────────────────────────
for param in model.lm.parameters():
    param.requires_grad = True
model.lm.train()

# Verify parameter counts
lm_trainable = sum(p.numel() for p in model.lm.parameters() if p.requires_grad)
encodec_frozen = sum(p.numel() for p in model.compression_model.parameters() if not p.requires_grad)

print(f"\nTrainable parameters (LM):    {lm_trainable/1e6:.1f}M")
print(f"Frozen parameters (EnCodec):  {encodec_frozen/1e6:.1f}M")

# ─── Optimizer ─────────────────────────────────────────────────────────
# AdamW = Adam with weight decay (helps prevent overfitting)
# Only pass LM parameters — EnCodec is deliberately excluded
optimizer = AdamW(
    model.lm.parameters(),
    lr=LEARNING_RATE,
    betas=(0.9, 0.95),   # Standard transformer optimizer settings
    weight_decay=0.01,   # L2 regularization — helps prevent memorization
)

# Learning rate scheduler — slowly decreases LR toward end of training
# This helps the model settle into a good minimum rather than bouncing around
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

print(f"\nOptimizer: AdamW, LR={LEARNING_RATE}, weight_decay=0.01")
print(f"Scheduler: CosineAnnealingLR over {NUM_EPOCHS} epochs")

## Step 4: The Training Loop

Here's the core logic of what happens each training step:

```
For each batch of (audio, description) pairs:

  1. [EnCodec, frozen]  audio_tensor → codes  (audio → integer tokens)
  2. [T5, frozen]       descriptions → text_embeddings
  3. [LM, training]     (codes, text_embeddings) → predicted_codes
  4.                    loss = cross_entropy(predicted_codes, actual_next_codes)
  5.                    loss.backward()  → compute gradients for LM
  6.                    optimizer.step() → update LM weights
```

The LM is doing **next-token prediction** (same as GPT), just for audio tokens instead of text tokens. It sees all previous tokens + the text prompt, and predicts the next one. The loss is the cross-entropy between its prediction and the actual next token.

In [ ]:
def prepare_conditions(descriptions: list) -> list:
    """
    Convert text description strings to AudioCraft's ConditioningAttributes.
    This is what the T5 text encoder uses as input.
    """
    return [
        ConditioningAttributes(text={'description': desc})
        for desc in descriptions
    ]


def compute_cross_entropy_loss(logits, codes, mask):
    """
    Compute cross-entropy loss for next-token prediction.
    
    Args:
        logits: [B, K, T, card] - predicted logits for each codebook position
        codes: [B, K, T] - ground truth token IDs
        mask: [B, K, T] - mask for valid positions
    
    Returns:
        Scalar loss tensor
    """
    B, K, T, card = logits.shape
    
    # Reshape for cross_entropy: (B*K*T, card) vs (B*K*T,)
    logits_flat = logits.reshape(-1, card)
    codes_flat = codes.reshape(-1)
    mask_flat = mask.reshape(-1)
    
    # Check for NaN in logits (debugging)
    if torch.isnan(logits_flat).any():
        nan_count = torch.isnan(logits_flat).sum().item()
        print(f"  WARNING: {nan_count} NaN values in logits!")
        return torch.tensor(float('nan'), device=logits.device)
    
    # Compute per-token cross entropy
    loss_per_token = F.cross_entropy(logits_flat, codes_flat, reduction='none')
    
    # Apply mask and compute mean over valid positions
    masked_loss = loss_per_token * mask_flat.float()
    loss = masked_loss.sum() / (mask_flat.sum() + 1e-8)
    
    return loss


def train_one_epoch(model, dataloader, optimizer, device, grad_clip):
    """
    Run one full pass over the training data.
    Returns: average loss for this epoch.
    """
    model.lm.train()
    model.compression_model.eval()  # Always keep EnCodec in eval mode

    total_loss = 0.0
    num_batches = 0
    nan_batches = 0

    for audio_batch, descriptions in dataloader:
        # audio_batch: [B, 1, T] — batch of mono audio tensors
        audio_batch = audio_batch.to(device)

        # ── Step 1: Tokenize audio with frozen EnCodec ──────────────────
        # No gradients here — EnCodec is frozen, and we don't need grads
        # through it anyway (codes are integers, not differentiable)
        with torch.no_grad():
            codes, scale = model.compression_model.encode(audio_batch)
            # codes: [B, K, T] where K=4 codebooks, T=time steps

        # ── Step 2: Prepare text conditioning ──────────────────────────
        conditions = prepare_conditions(descriptions)

        # ── Step 3: Forward pass through LM + compute loss ─────────────
        # DISABLED autocast for stability after dtype fix
        lm_output = model.lm.compute_predictions(
            codes=codes,
            conditions=conditions,
        )
        # Compute cross-entropy loss manually
        loss = compute_cross_entropy_loss(lm_output.logits, codes, lm_output.mask)

        # Skip batch if NaN
        if torch.isnan(loss):
            nan_batches += 1
            if nan_batches <= 3:
                print(f"  Skipping batch due to NaN (batch {num_batches + nan_batches})")
            continue

        # ── Step 4: Backward pass ───────────────────────────────────────
        optimizer.zero_grad()
        loss.backward()

        # Gradient clipping — prevents the optimizer from taking huge steps
        # on noisy batches (stabilizes training)
        torch.nn.utils.clip_grad_norm_(model.lm.parameters(), grad_clip)

        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

    if nan_batches > 0:
        print(f"  Total NaN batches skipped: {nan_batches}")
    
    return total_loss / num_batches if num_batches > 0 else float('nan')


@torch.no_grad()
def evaluate(model, dataloader, device):
    """
    Evaluate on validation set — no gradient computation.
    Returns: average validation loss.
    """
    model.lm.eval()
    model.compression_model.eval()

    total_loss = 0.0
    num_batches = 0

    for audio_batch, descriptions in dataloader:
        audio_batch = audio_batch.to(device)

        codes, scale = model.compression_model.encode(audio_batch)
        conditions = prepare_conditions(descriptions)

        # DISABLED autocast for stability
        lm_output = model.lm.compute_predictions(codes=codes, conditions=conditions)
        loss = compute_cross_entropy_loss(lm_output.logits, codes, lm_output.mask)
        
        if not torch.isnan(loss):
            total_loss += loss.item()
            num_batches += 1

    model.lm.train()  # Switch back to train mode
    return total_loss / num_batches if num_batches > 0 else float('nan')


print("Training functions defined (autocast DISABLED for stability)")
print("\nReady to train. What you'll see each epoch:")
print("  train_loss: Should decrease - model is learning")
print("  val_loss:   Should also decrease (or stay flat) - model is generalizing")
print("  If val_loss rises while train_loss falls -> overfitting, stop early!")

## Step 5: Run Training

This is it. A few things to watch:
- **Initial loss** (~5-8 is typical for random text predictions over 4 codebooks × 2048 vocab)
- **Convergence** — the loss should drop within the first 5 epochs if the data is clean
- **Early stopping** — if val loss hasn't improved for 10 epochs, consider stopping

On a 5090, this should run at ~5-15 seconds per epoch for a 100-clip dataset.

In [ ]:
import time

train_losses = []
val_losses   = []
best_val_loss = float('inf')
best_epoch    = 0
no_improve_count = 0
EARLY_STOP_PATIENCE = 10  # Stop if val loss doesn't improve for this many epochs

print(f"Starting training: {NUM_EPOCHS} epochs, batch size {BATCH_SIZE}")
print(f"{'Epoch':>6} {'Train Loss':>12} {'Val Loss':>10} {'LR':>10} {'Time':>8} {'Note':<15}")
print('─' * 70)

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    # Training pass
    train_loss = train_one_epoch(model, train_loader, optimizer, DEVICE, GRAD_CLIP)
    train_losses.append(train_loss)

    # Validation pass
    val_loss = evaluate(model, val_loader, DEVICE)
    val_losses.append(val_loss)

    # Scheduler step
    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]

    elapsed = time.time() - t0
    note = ""

    # Track best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        no_improve_count = 0
        note = "✅ best"
        # Save best model checkpoint
        torch.save({
            'epoch': epoch,
            'lm_state_dict': model.lm.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
        }, CHECKPOINT_DIR / 'best_checkpoint.pt')
    else:
        no_improve_count += 1

    # Periodic checkpoint
    if epoch % SAVE_EVERY == 0:
        torch.save({
            'epoch': epoch,
            'lm_state_dict': model.lm.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
        }, CHECKPOINT_DIR / f'checkpoint_epoch{epoch:03d}.pt')
        if not note:
            note = "💾 saved"

    print(f"{epoch:>6} {train_loss:>12.4f} {val_loss:>10.4f} {current_lr:>10.2e} {elapsed:>7.1f}s {note:<15}")

    # Early stopping
    if no_improve_count >= EARLY_STOP_PATIENCE:
        print(f"\n⏹️  Early stopping at epoch {epoch} (no val improvement for {EARLY_STOP_PATIENCE} epochs)")
        print(f"   Best was epoch {best_epoch} with val_loss={best_val_loss:.4f}")
        break

print(f"\n✅ Training complete! Best checkpoint: epoch {best_epoch}, val_loss={best_val_loss:.4f}")
print(f"   Saved to: {CHECKPOINT_DIR / 'best_checkpoint.pt'}")

## Step 6: Visualize Training

The loss curves tell you a lot:
- **Both declining together**: Great — the model is learning and generalizing
- **Train falling, val rising**: Overfitting — model is memorizing clips, not learning style  
- **Both stuck high**: Learning rate might be too low, or data quality issues
- **Loss explodes**: Gradient clipping should prevent this, but if it happens, reduce LR

In [ ]:
epochs_run = list(range(1, len(train_losses) + 1))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
ax1.plot(epochs_run, train_losses, 'b-o', markersize=3, label='Train Loss', linewidth=2)
ax1.plot(epochs_run, val_losses,   'r-o', markersize=3, label='Val Loss',   linewidth=2)
ax1.axvline(x=best_epoch, color='green', linestyle='--', alpha=0.7, label=f'Best (epoch {best_epoch})')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.set_title('Training Progress')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Train-val gap (overfitting indicator)
gaps = [v - t for t, v in zip(train_losses, val_losses)]
ax2.plot(epochs_run, gaps, 'purple', linewidth=2)
ax2.axhline(y=0, color='black', linestyle='-', alpha=0.3)
ax2.fill_between(epochs_run, gaps, 0,
                  where=[g > 0 for g in gaps], alpha=0.2, color='red',
                  label='Overfitting zone')
ax2.fill_between(epochs_run, gaps, 0,
                  where=[g <= 0 for g in gaps], alpha=0.2, color='green',
                  label='Underfitting zone')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Val Loss − Train Loss')
ax2.set_title('Overfitting Indicator')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('checkpoints/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary table
print(f"\nTraining Summary:")
print(f"  Initial train loss:  {train_losses[0]:.4f}")
print(f"  Final train loss:    {train_losses[-1]:.4f}  (change: {train_losses[-1]-train_losses[0]:+.4f})")
print(f"  Best val loss:       {best_val_loss:.4f} at epoch {best_epoch}")
print(f"  Total improvement:   {((train_losses[0] - train_losses[-1]) / train_losses[0] * 100):.1f}%")

## Step 7: Quick Sanity Check — Generate from the Fine-Tuned Model

Before moving to the evaluation notebook, let's do a quick test: generate one clip from the fine-tuned model and listen. Does it sound more like throat singing than the baseline?

In [ ]:
import soundfile as sf
import IPython.display as ipd

# Load the best checkpoint's LM weights
checkpoint = torch.load(CHECKPOINT_DIR / 'best_checkpoint.pt', map_location=DEVICE)
model.lm.load_state_dict(checkpoint['lm_state_dict'])
model.lm.eval()

print(f"Loaded best checkpoint (epoch {checkpoint['epoch']}, val_loss={checkpoint['val_loss']:.4f})")

# Set generation params
model.set_generation_params(
    duration=10,
    use_sampling=True,
    top_k=250,
    temperature=1.0,
    cfg_coef=3.0,
)

test_prompts = [
    "tuvan throat singing, khoomei, drone with overtone melody",
    "mongolian throat singing, meditative, ancient",
]

print("\nGenerating sanity check clips...")
with torch.no_grad():
    wav_ft = model.generate(test_prompts)

for i, (prompt, audio) in enumerate(zip(test_prompts, wav_ft)):
    audio_np = audio.squeeze(0).cpu().float().numpy()
    filename = f"checkpoints/sanity_{i+1}.wav"
    sf.write(filename, audio_np, model.sample_rate)
    print(f"\n🎵 \"{prompt}\"")
    display(ipd.Audio(audio_np, rate=model.sample_rate))

print("\n✅ Sanity check complete.")
print("   ➡️  For full evaluation and comparison, open Notebook 4.")

## Notebook 3 Summary

**What we did:**
- Froze EnCodec (audio tokenizer) — it's not what needs to change
- Trained only the transformer LM — the part that "decides" which tokens to generate next
- Used a conservative LR (1e-5) to preserve the model's existing music knowledge
- Monitored overfitting with train/val loss curves
- Saved checkpoints so we can reload the best version

**Things to try if results aren't satisfying:**
- More data (ideally 200+ clips)
- Slightly higher LR (5e-5) for more dramatic adaptation
- Train for more epochs with lower LR
- Try musicgen-medium (1.5B) — more capacity to learn new styles
- More diverse text descriptions

**The key question we'll answer in Notebook 4:** Did the model actually learn throat singing, or just get slightly confused?

---

**➡️ Next: `04_evaluation.ipynb` — Compare base model vs. fine-tuned model**